# Teaching CrewAI: Sequential and Parallel Workflows

## What this notebook covers

By the end of this lesson, you will be able to:

1. Load credentials from a local `.env` file, with hidden `getpass` prompts for missing keys.
2. Uses Gemini (or OpenAI model).
3. Build a custom Tavily web-search tool.
4. Create agents, tasks, and a crew.
5. Run a **sequential (series)** research-to-writing workflow.
6. Run **independent tasks concurrently** and inspect both outputs.
7. Use notebook-friendly async kickoff options.

Run cells from top to bottom. Kickoff cells make real API calls and may incur charges. This is a workflow lesson; verify healthcare claims and sources before reusing generated text.

## 1. Install dependencies

Run this once in your notebook kernel, then restart the kernel if requested. The LiteLLM extra supports the Azure route used here. Provider APIs evolve; record the installed versions when teaching and freeze a tested environment for repeat classes.

In [9]:
%pip install "crewai[google-genai]==1.15.21" requests python-dotenv

  Using cached google_genai-1.65.0-py3-none-any.whl.metadata (51 kB)
Using cached google_genai-1.65.0-py3-none-any.whl (724 kB)
  Attempting uninstall: google-genai
    Found existing installation: google-genai 2.28.0
    Uninstalling google-genai-2.28.0:
      Successfully uninstalled google-genai-2.28.0
Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-google-genai 4.4.0 requires google-genai<3.0.0,>=2.20.0, but you have google-genai 1.65.0 which is incompatible.

[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
%pip install "litellm==1.105.1"

   ---------------------------------------- 0.0/39.8 MB ? eta -:--:--
   -- ------------------------------------- 2.6/39.8 MB 13.7 MB/s eta 0:00:03
   -------- ------------------------------- 8.7/39.8 MB 21.5 MB/s eta 0:00:02
   --------------- ------------------------ 15.2/39.8 MB 24.5 MB/s eta 0:00:02
   --------------------- ------------------ 21.8/39.8 MB 26.4 MB/s eta 0:00:01
   -------------------------- ------------- 26.2/39.8 MB 25.5 MB/s eta 0:00:01
   ------------------------------- -------- 31.2/39.8 MB 25.4 MB/s eta 0:00:01
   ------------------------------------- -- 37.2/39.8 MB 26.0 MB/s eta 0:00:01
   ---------------------------------------- 39.8/39.8 MB 25.3 MB/s  0:00:01
  Attempting uninstall: litellm
    Found existing installation: litellm 1.106.0
    Uninstalling litellm-1.106.0:
      Successfully uninstalled litellm-1.106.0
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


crewai-tools 1.15.21 has requirement tiktoken<0.13,>=0.8.0, but you have tiktoken 0.14.0.
Note: you may need to restart the kernel to use updated packages.


In [18]:
# Check package version

import importlib.metadata

# List the distribution package names
packages = ["crewai", "litellm", "requests", "python-dotenv", "crewai-tools"]

for package in packages:
    try:
        version = importlib.metadata.version(package)
        print(f"{package} version: {version}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{package} is not installed in this environment.")

crewai version: 1.15.21
litellm version: 1.105.1
requests version: 2.34.2
python-dotenv version: 1.2.3
crewai-tools version: 1.15.21


## 2. Imports and version information

An **agent** has a role and tools, a **task** defines work and its expected output, and a **crew** coordinates tasks.

In [7]:
import os
import json
from datetime import date
from pathlib import Path
from getpass import getpass
from dotenv import load_dotenv
from crewai import LLM

load_dotenv(Path.cwd() / ".env", override=False)

import requests
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from crewai import Agent, Task, Crew, Process
# from crewai.llm import LLM
from crewai.tools import BaseTool

crewai: 1.15.21
litellm: 1.105.1
requests: 2.34.2
python-dotenv: 1.2.3


## 3. Credentials: .env first and then  hidden prompts seconds

In [8]:
tavily_api_key = os.getenv("TAVILY_API_KEY", "").strip()

if not tavily_api_key:
    tavily_api_key = getpass("Enter your Tavily API key: ").strip()

os.environ["TAVILY_API_KEY"] = tavily_api_key

print("Credential loading is ready.")

Credential loading is ready.


## 4. LLM configuration

In [2]:
# OPTION 1: Gemini — active by default
gemini_api_key = os.getenv("GEMINI_API_KEY", "").strip()
if not gemini_api_key:
    gemini_api_key = getpass("Enter your Gemini API key: ").strip()

gemini_model = os.getenv("GEMINI_MODEL", "gemini-3.5-flash-lite").strip()


llm = LLM(
    model=f"gemini/{gemini_model}",
    api_key=gemini_api_key,
    is_litellm=False,
    timeout=30,
    max_retries=0,
)


# OPTION 2: OpenAI
# Comment out the Gemini block above and uncomment this block.
#
# openai_api_key = os.getenv("OPENAI_API_KEY", "").strip()
# if not openai_api_key:
#     openai_api_key = getpass("Enter your OpenAI API key: ").strip()
#
# if not openai_api_key:
#     raise ValueError("OpenAI API key cannot be empty.")
#
# llm = LLM(
#     model=f"openai/{os.getenv('OPENAI_MODEL', 'gpt-4o-mini')}",
#     api_key=openai_api_key,
#     is_litellm=False,
#     timeout=30,
#     max_retries=0,
# )

In [5]:
# Make a minimal authenticated request using the configured API key.

try:
    response = llm.call(
        messages=[
            {"role": "user", "content": "Reply with only: OK"}
        ]
    )

    if not response:
        print("The request completed but returned no text.")

    print("LLM connection successful.")
    print("Response:", response)
    

except Exception as exc:
    # Avoid printing request details that might contain credentials.
    print(f"LLM check failed: {type(exc).__name__}")
    print("Check the API key, model ID, network access, and account quota.")
    

LLM connection successful.
Response: OK


## 5. A custom Tavily search tool

`BaseTool` exposes a callable action to an agent. Its schema describes the input. Returning snippets along with titles and URLs gives the researcher evidence to summarize, rather than links alone. Snippets are incomplete evidence and may omit publication dates.

The HTTP request has a timeout and checks for failures. The error message excludes request details to avoid exposing credentials. The instruction to search only once is a prompt instruction, not a programmatic call limit.

In [9]:
class SearchInput(BaseModel):
    query: str = Field(..., description="The web search query.")


class TavilySearchTool(BaseTool):
    name: str = "web_search"
    description: str = "Search the web for recent information and source snippets."
    args_schema: type[BaseModel] = SearchInput

    def _run(self, query: str) -> str:
        try:
            response = requests.post(
                "https://api.tavily.com/search",
                json={
                    "api_key": os.environ["TAVILY_API_KEY"],
                    "query": query,
                    "max_results": 5,
                },
                timeout=30,
            )
            response.raise_for_status()
            data = response.json()
        except (requests.RequestException, ValueError):
            raise RuntimeError(
                "Tavily search failed. Check credentials, connectivity, and quota."
            ) from None

        results = [
            {
                "title": item.get("title", "No title"),
                "url": item.get("url", ""),
                "snippet": item.get("content", ""),
            }
            for item in data.get("results", [])
        ]
        return json.dumps(results, ensure_ascii=False) if results else "No results found."


search_tool = TavilySearchTool()

## 6. Define the agents

The researcher can search the web. The writer has no search tool and works from its task context or existing model knowledge. Delegation is disabled to keep the demonstration easy to follow. A factory gives each example fresh agents while sharing the chosen model configuration.

In [10]:
def make_agents():
    researcher = Agent(
        role="AI Researcher",
        goal="Find recent, source-backed advancements in AI for healthcare",
        backstory="You research AI in healthcare and distinguish evidence from speculation.",
        verbose=True,
        allow_delegation=False,
        max_iter=6,
        llm=llm,
        tools=[search_tool],
    )
    writer = Agent(
        role="Technical Writer",
        goal="Explain healthcare AI clearly to executives",
        backstory="You summarize research accurately and preserve source references.",
        verbose=True,
        allow_delegation=False,
        llm=llm,
    )
    return researcher, writer


TODAY = date.today().isoformat()

# Serial 

## 7. Sequential (series) tasks

**Research → executive summary.** The writer needs the research output, so the tasks run in order. `context=[task_research]` makes that dependency explicit. The research prompt asks for 5–6 sentences covering three advancements; the summary targets about 200 words.

In [12]:
researcher, writer = make_agents()

task_research = Task(
    description=(
        f"Today is {TODAY}. Search once for three recent advancements in AI for healthcare "
        "from the past 12 months. After receiving results, do not call tools again. "
        "Explain the three advancements in 5–6 sentences, with source URLs and dates "
        "where available. Do not invent evidence; state if the results are insufficient."
    ),
    expected_output="Source-backed notes on three advancements, in 5–6 sentences.",
    agent=researcher,
)

task_write = Task(
    description=(
        "Write an executive summary of about 200 words using only the research notes. "
        "Preserve source URLs and limitations. Do not add unsupported claims."
    ),
    expected_output="An executive summary of three healthcare AI advancements.",
    agent=writer,
    context=[task_research],
)

crew_serial = Crew(
    agents=[researcher, writer],
    tasks=[task_research, task_write],
    process=Process.sequential,
    verbose=True,
)

### Run the sequential crew

This cell starts the first paid model/search workflow. If synchronous kickoff produces an event-loop error in your notebook environment, replace the kickoff line with `serial_result = await crew_serial.kickoff_async()`.

In [13]:
# serial_result = crew_serial.kickoff()
serial_result = await crew_serial.kickoff_async() # use this for notebook
print(serial_result.raw)

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.15.21                                                                                       │
│  Latest version:  1.15.34                                                                                       │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 3316e631-6c4f-4104-941e-3af03cdc4fcf                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Today is 2026-11-03. Search once for three recent advancements in AI for healthcare from the past 12     │
│  months. After receiving results, do not call tools again. Explain the three advancements in 5–6 sentences,     │
│  with source URLs and dates where available. Do not invent evidence; state if the results are insufficient.     │
│  ID: a1321b83-e017-403f-9bad-5d62263f35f0                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: AI Researcher                                                                                           │
│                                                                                                                 │
│  Task: Today is 2026-11-03. Search once for three recent advancements in AI for healthcare from the past 12     │
│  months. After receiving results, do not call tools again. Explain the three advancements in 5–6 sentences,     │
│  with source URLs and dates where available. Do not invent evidence; state if the results are insufficient.     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: web_search                                                                                               │
│  Args: {'query': 'AI healthcare advancements clinical trials FDA approval 2025 2026'}                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool web_search executed with result: [{"title": "List of Artificial Intelligence-Enabled Medical Devices | FDA", "url": "https://www.fda.gov/medical-devices/artificial-intelligence-enabled-medical-devices/list-artificial-intelligence-ena...


... (8 outputs left out)

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Write an executive summary of about 200 words using only the research notes. Preserve source URLs and    │
│  limitations. Do not add unsupported claims.                                                                    │
│  Agent: Technical Writer                                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 3316e631-6c4f-4104-941e-3af03cdc4fcf                                                                       │
│  Final Output: Here is the executive summary of recent healthcare AI advancements:                              │
│                                                                                                                 │
│  Recent developments in healthcare artificial intelligence highlight three major areas of progress:             │
│                                                                                                                 │
│  1. **Clinical Generative AI:** Generative AI tools are now widely adopted in clinical settings to              │
│  automatically draft patient visit notes. This significantly cuts down physician administrative time and helps  │
│  mitigate burnout, as noted in the Stanford HAI 2026 AI Index Report                                            │
│  (https://hai.stanford.edu/ai-index/2026-ai-index-report/medicine).                                             │
│  2. **Genomic and Cellular Modeling:** Foundational modeling frontiers advanced notably with major 2025         │
│  releases like Evo 2 and AlphaGenome. These tools model genomic architecture and cellular functions at          │
│  unprecedented scales.                                                                                          │
│  3. **Regulatory Approvals:** Regulatory bodies continue incorporating software advancements, with the FDA      │
│  authorizing hundreds of new AI-enabled medical devices—primarily focused on radiology and cardiac              │
│  diagnostics—according to the official FDA database                                                             │
│  (https://www.fda.gov/medical-devices/artificial-intelligence-enabled-medical-devices/list-artificial-intellig  │
│  ence-enabled-medical-devices).                                                                                 │
│                                                                                                                 │
│  *Limitations: This summary is strictly limited to the provided research notes and source materials.*           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Here is the executive summary of recent healthcare AI advancements:

Recent developments in healthcare artificial intelligence highlight three major areas of progress:

1. **Clinical Generative AI:** Generative AI tools are now widely adopted in clinical settings to automatically draft patient visit notes. This significantly cuts down physician administrative time and helps mitigate burnout, as noted in the Stanford HAI 2026 AI Index Report (https://hai.stanford.edu/ai-index/2026-ai-index-report/medicine).
2. **Genomic and Cellular Modeling:** Foundational modeling frontiers advanced notably with major 2025 releases like Evo 2 and AlphaGenome. These tools model genomic architecture and cellular functions at unprecedented scales. 
3. **Regulatory Approvals:** Regulatory bodies continue incorporating software advancements, with the FDA authorizing hundreds of new AI-enabled medical devices—primarily focused on radiology and cardiac diagnostics—according to the official FDA database (http

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

# PARALLEL

## 8. Parallel tasks

**Drug-discovery company search ∥ diagnostics report.** These tasks are independent: the diagnostics writer does not consume the company list. The first task has `async_execution=True`, allowing it to overlap with the next task. An explicit empty context keeps the second task independent.

The crew still uses `Process.sequential` to schedule its task list; there is no `Process.parallel` setting here. Task-level async execution enables the overlap. By contrast, async **kickoff** controls how the caller waits for the whole crew.

The diagnostics report uses the model's existing knowledge; it is not a fresh web-researched report.

In [14]:
parallel_researcher, parallel_writer = make_agents()

task_parallel_1 = Task(
    description=(
        "Search the web for five AI companies focused on drug discovery. "
        "Give one short line per company about its specialization and a source URL."
    ),
    expected_output="Five company names, specializations, and source URLs.",
    async_execution=True,
    agent=parallel_researcher,
    context=[],
)

task_parallel_2 = Task(
    description=(
        "Write about 200 words on how AI is transforming patient diagnostics. "
        "Include examples and limitations. Use general knowledge and avoid claims "
        "about the latest approvals or performance figures without evidence."
    ),
    expected_output="A short explanatory report on AI in patient diagnostics.",
    agent=parallel_writer,
    context=[],
)

crew_parallel = Crew(
    agents=[parallel_researcher, parallel_writer],
    tasks=[task_parallel_1, task_parallel_2],
    process=Process.sequential,
    verbose=True,
)



### Run the parallel crew: notebook async note

The original line is `parallel_result = crew_parallel.kickoff()`. **It may not work in some notebook environments**, for example when an installed execution path conflicts with the notebook's running event loop. In that case use an async option.

The runnable cell below uses **`await crew_parallel.kickoff_async()`**, a thread-based wrapper for synchronous kickoff. Newer CrewAI versions also offer native async via **`await crew_parallel.akickoff()`**. Choose one method per run; running both repeats the API calls. Use top-level `await` in Jupyter instead of `asyncio.run()`.

Async kickoff does not itself make dependent tasks parallel. The task flags and context dependencies above determine concurrency. See the [CrewAI kickoff methods](https://docs.crewai.com/en/concepts/crews) and [task execution documentation](https://docs.crewai.com/en/concepts/tasks).

In [15]:
# Original synchronous option (use instead of the await line if desired):
# parallel_result = crew_parallel.kickoff()

parallel_result = await crew_parallel.kickoff_async()

# Alternative for CrewAI versions supporting native async:
# parallel_result = await crew_parallel.akickoff()

print(parallel_result.raw)

# NOTE:
# 1) CrewAI starts task_parallel_1 asynchronously.
# 2) It moves on to task_parallel_2 while task 1 can still be running.
# 3) It waits for both tasks to finish before returning the crew result.

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.15.21                                                                                       │
│  Latest version:  1.15.34                                                                                       │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 410243a2-d1fc-4493-9d1e-2704d67d1f18                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Search the web for five AI companies focused on drug discovery. Give one short line per company about    │
│  its specialization and a source URL.                                                                           │
│  ID: aac5733b-a55a-4511-b797-c2850f1a4e45                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: AI Researcher                                                                                           │
│                                                                                                                 │
│  Task: Search the web for five AI companies focused on drug discovery. Give one short line per company about    │
│  its specialization and a source URL.                                                                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#2) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: web_search                                                                                               │
│  Args: {'query': 'top AI drug discovery companies 2024 2025'}                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool web_search executed with result: [{"title": "11 Innovative Companies Using AI for Drug Discovery", "url": "https://wewillcure.com/insights/company-profiles/ai-and-machine-learning/innovative-companies-using-ai-for-drug-discovery", "s...

... (8 outputs left out)

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Write about 200 words on how AI is transforming patient diagnostics. Include examples and limitations.   │
│  Use general knowledge and avoid claims about the latest approvals or performance figures without evidence.     │
│  Agent: Technical Writer                                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 410243a2-d1fc-4493-9d1e-2704d67d1f18                                                                       │
│  Final Output: Artificial intelligence (AI) is fundamentally transforming patient diagnostics by enhancing      │
│  speed, accuracy, and consistency across medical imaging and pathology. Machine learning algorithms excel at    │
│  rapidly processing vast amounts of clinical data, enabling healthcare providers to identify complex patterns   │
│  that the human eye might miss.                                                                                 │
│                                                                                                                 │
│  **Key Applications and Examples**                                                                              │
│  AI is most prominent in radiology, where algorithms assist in screening routine scans for anomalies such as    │
│  early-stage tumors, micro-fractures, or pulmonary nodules. In ophthalmology, AI tools analyze retinal scans    │
│  to detect diabetic retinopathy years before traditional symptoms worsen. Furthermore, in pathology, digital    │
│  tools help quantify biomarker expression in biopsy slides, reducing inter-observer variability and             │
│  accelerating oncology workflows.                                                                               │
│                                                                                                                 │
│  **Current Limitations**                                                                                        │
│  Despite its potential, AI diagnostics face significant hurdles. Models trained on homogeneous data often       │
│  suffer from bias, leading to reduced accuracy when applied to diverse patient populations. Additionally, the   │
│  "black box" nature of deep learning makes it difficult for clinicians to understand how an algorithm reached   │
│  a specific conclusion, complicating accountability. Integration into existing electronic health record         │
│  systems remains cumbersome, and regulatory frameworks are still evolving to ensure continuous safety as        │
│  software updates occur.                                                                                        │
│                                                                                                                 │
│  Ultimately, AI serves as a powerful clinical decision-support tool rather than a replacement for human         │
│  expertise.                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Artificial intelligence (AI) is fundamentally transforming patient diagnostics by enhancing speed, accuracy, and consistency across medical imaging and pathology. Machine learning algorithms excel at rapidly processing vast amounts of clinical data, enabling healthcare providers to identify complex patterns that the human eye might miss. 

**Key Applications and Examples**
AI is most prominent in radiology, where algorithms assist in screening routine scans for anomalies such as early-stage tumors, micro-fractures, or pulmonary nodules. In ophthalmology, AI tools analyze retinal scans to detect diabetic retinopathy years before traditional symptoms worsen. Furthermore, in pathology, digital tools help quantify biomarker expression in biopsy slides, reducing inter-observer variability and accelerating oncology workflows.

**Current Limitations**
Despite its potential, AI diagnostics face significant hurdles. Models trained on homogeneous data often suffer from bias, leading to reduced a

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

### Inspect both task outputs

`parallel_result.raw` represents the final task's output, not a combined report of all tasks. Inspect `tasks_output` to see the company list as well as the diagnostics report.

In [16]:
for label, output in zip(
    ["Drug-discovery companies", "Patient diagnostics"],
    parallel_result.tasks_output,
):
    print(f"\n=== {label} ===\n")
    print(output.raw)


=== Drug-discovery companies ===

1. **Insilico Medicine**: Utilizes a proprietary generative AI platform (Pharma.AI) spanning target discovery, small molecule design, and clinical trial outcome prediction. [Insilico Medicine](https://insilico.com/)
2. **Recursion Pharmaceuticals**: Integrates automated high-throughput biology with machine learning and phenomics to map biology and discover novel therapeutics. [Recursion](https://www.recursion.com/)
3. **Schrödinger**: Employs physics-based computational methods combined with machine learning to accelerate drug discovery and optimize small-molecule design. [Schrödinger](https://www.schrodinger.com/)
4. **Isomorphic Labs**: Applies advanced AI models, leveraging AlphaFold technology, to decode biological structures and transform drug design. [Isomorphic Labs](https://www.isomorphiclabs.com/)
5. **Insitro**: Combines machine learning with high-throughput experimental biology to generate disease-relevant cellular data for target discovery

## 9. Teaching discussion and reuse

- **Dependency:** Why must the sequential writer wait, while the diagnostics writer can work independently?
- **Evidence:** Which output has web sources? Which relies on model knowledge?
- **Concurrency:** API latency and rate limits affect overlap; parallel execution does not guarantee a speedup.
- **Model comparison:** Change provider/deployment settings, restart, and rerun. Compare accuracy, source use, latency, and cost using the same tasks.
- **Extension:** Add a final synthesis task with `context=[task_parallel_1, task_parallel_2]` if you want a combined report that waits for both outputs.

